## Text input

In [24]:
from langchain.agents import create_agent
from azure_openai_llm import create_azure_llm
from langchain.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv()

llm = create_azure_llm()

agent = create_agent(llm,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [25]:
question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Welcome to Lunara, the shining capital city of The Moon!

Nestled within the vast expanse of the Sea of Tranquility, Lunara is a marvel of human ingenuity and extraterrestrial architecture. The city is built inside a massive transparent dome that filters solar radiation while allowing breathtaking views of the Earth hanging in the black sky above.

Lunara’s skyline is a blend of sleek silver spires and bioengineered crystal towers that shimmer with soft hues of blue and violet. The city is powered by a network of fusion reactors and solar farms scattered across the lunar surface, ensuring sustainable energy for its inhabitants.

The city is divided into several districts:

- **Helios Plaza:** The administrative heart, where the Lunar Council convenes in the grand Orbital Chamber, a rotating structure symbolizing unity and progress.

- **Selene Gardens:** A vast network of biospheres housing Earth plants and trees, creating green spaces for recreation and research on lunar agriculture.


## Image input

In [27]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)
print(uploader.value)

FileUpload(value=(), accept='.png', description='Upload')

()


In [28]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [29]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this diagram"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

This diagram illustrates the interaction flow between a bot (represented by the robot icon) and a system composed of three main components: Model, Tools, and Result.

- The process starts with a **Request** received by the **Model** (purple box).
- The **Model** then interacts with **Tools** (red box) by sending **Actions** and receiving **Observations** in return. This interaction is shown within a dashed box, indicating a feedback loop between the Model and Tools.
- After processing the information from the Tools, the Model produces a **Result** (gray box), which is the final output of the system.
- The initial request flows into the Model, which manages the decision-making and tool usage, while the result is the outcome delivered back to the bot or user.

In summary, this diagram represents a system where a Model interprets requests, uses Tools to gather information or perform actions, processes observations, and finally generates a Result. The interaction between Model and Tools is

In [34]:
response['messages'][-1].response_metadata

{'token_usage': {'completion_tokens': 211,
  'prompt_tokens': 1300,
  'total_tokens': 1511,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_provider': 'openai',
 'model_name': 'gpt-4.1-mini-2025-04-14',
 'system_fingerprint': 'fp_b6f445fc1c',
 'id': 'chatcmpl-DOH7ZyH8Wjf2tp0cHGhzbu6i3HCjl',
 'prompt_filter_results': [{'prompt_index': 0,
   'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'},
    'jailbreak': {'filtered': False, 'detected': False},
    'self_harm': {'filtered': False, 'severity': 'safe'},
    'sexual': {'filtered': False, 'severity': 'safe'},
    'violence': {'filtered': False, 'severity': 'safe'}}}],
 'finish_reason': 'stop',
 'logprobs': None,
 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'},
  'protected_material_code': {'filtered': False, 'd

In [ ]:
# Check what deployments you have available
import os
from dotenv import load_dotenv

load_dotenv()

print("🔍 Current Azure AI Foundry Configuration:")
print("=" * 50)
print(f"Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"Chat Deployment: {os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}")
print(f"Audio Deployment: {os.getenv('AZURE_OPENAI_AUDIO_DEPLOYMENT')}")
print(f"API Version: {os.getenv('AZURE_OPENAI_API_VERSION')}")
print("=" * 50)

# Test if we can create connections to both deployments
print("\n📋 Testing Model Availability:")
print("-" * 30)

try:
    from azure_openai_llm import create_azure_llm
    
    print("✅ Chat model connection:")
    chat_llm = create_azure_llm(type="chat")
    print(f"   Model type: {type(chat_llm).__name__}")
    
    print("\n✅ Audio model connection:")  
    audio_llm = create_azure_llm(type="audio")
    print(f"   Model type: {type(audio_llm).__name__}")
    
except Exception as e:
    print(f"❌ Error: {e}")

In [9]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm # tqdm provides a visual progress bar during the recording loop.

# Steps:
# 2) Record audio from the microphone using sounddevice (sd).
# 3) Display a short progress bar during recording.
# 4) Wait for recording completion to ensure data is finalized.
# 5) Convert WAV bytes to Base64 encoded string.

# Recording settings
duration = 3  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()
aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 30/30 [00:03<00:00,  9.89it/s]

Done.


In [10]:
from langchain.agents import create_agent
from azure_openai_llm import create_azure_llm
from langchain.messages import HumanMessage

agent = create_agent(create_azure_llm(type="audio", api="openai"))

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Follow the instructions in the audio and reply"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

Using audio deployment: lums-gpt-audio-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview
The sum of 2 plus 7 is 9.


In [18]:
response['messages'][-1].response_metadata

{'token_usage': {'completion_tokens': 12,
  'prompt_tokens': 49,
  'total_tokens': 61,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0,
   'text_tokens': 12},
  'prompt_tokens_details': {'audio_tokens': 30,
   'cached_tokens': 0,
   'image_tokens': 0,
   'text_tokens': 19}},
 'model_provider': 'openai',
 'model_name': 'gpt-audio-mini-2025-12-15',
 'system_fingerprint': 'fp_94129ed17c',
 'id': 'chatcmpl-DOH1jNUYNEjNV9EvZ4Tqoi7t5cmzj',
 'prompt_filter_results': [{'prompt_index': 0,
   'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'},
    'jailbreak': {'filtered': False, 'detected': False},
    'self_harm': {'filtered': False, 'severity': 'safe'},
    'sexual': {'filtered': False, 'severity': 'safe'},
    'violence': {'filtered': False, 'severity': 'safe'}}}],
 'finish_reason': 'stop',
 'logprobs': None,
 'content_filter_results': {'hate': {'filtered': False, 'seve